# Linux Lab 1: System Checking

My notes and command walkthrough for this lab.

Date: **2026-02-17**

Lab source: [KillerCoda Linux Lab 1 - System Checking](https://killercoda.com/het-tanis/course/Linux-Labs/01-system-checking)


## Task

**Task:** Determine the operating system values of a running Linux OS. Execute commands and evaluate output of a running Linux OS.

**Condition:** Given a Bash prompt, complete in the allotted 60 minute time.

**Standard:** Work through the lab at a pace that matches your learning goals.


## Scenario

I am a new system administrator at a company.

I was tasked with finding information about servers that have very little to no documentation. From a command prompt, I need to identify what type of system I am working on.

- What is the release and version of the system?
- What is the kernel version?
- How long has the system been up, and what kernel parameters were passed by GRUB at boot?


## Command Walkthrough and Output Log

This section keeps both the commands I used and a snapshot of their output for quick reference.


In [ ]:
# 1) Check Linux release/version
cat /etc/*release

# 2) Check kernel version
uname -r
cat /proc/version

# 3) Check uptime + load averages
uptime

# 4) Check boot/kernel parameters
cat /proc/cmdline


### Output Snapshot

```bash
$ cat /etc/*release

DISTRIB_ID=cachyos
DISTRIB_RELEASE="rolling"
DISTRIB_DESCRIPTION="CachyOS"
NAME="CachyOS Linux"
PRETTY_NAME="CachyOS"
ID=cachyos
ID_LIKE=arch
BUILD_ID=rolling
ANSI_COLOR="38;2;23;147;209"
HOME_URL="https://cachyos.org/"
DOCUMENTATION_URL="https://wiki.cachyos.org/"
SUPPORT_URL="https://discuss.cachyos.org/"
BUG_REPORT_URL="https://github.com/cachyos"
PRIVACY_POLICY_URL="https://terms.archlinux.org/docs/privacy-policy/"
LOGO=cachyos
```

```bash
$ uname -r
6.18.9-2-cachyos
```

```bash
$ cat /proc/version
Linux version 6.18.9-2-cachyos (linux-cachyos@cachyos) (clang version 21.1.6, LLD 21.1.6) #1 SMP PREEMPT_DYNAMIC Sat, 07 Feb 2026 00:59:36 +0000
```

```bash
$ uptime
 04:14:09 up 11:27,  1 user,  load average: 0.78, 0.71, 0.65
```

```bash
$ cat /proc/cmdline
BOOT_IMAGE=/@/boot/vmlinuz-linux-cachyos root=UUID=c3cbaaef-c7c7-4dca-bba5-3c398afad35f rw rootflags=subvol=@ nowatchdog nvme_load=YES zswap.enabled=0 splash loglevel=3
```


## Reflection: Questions and Answers

1. **What useful information do I see from `cat /etc/*release`? Why might I need it?**
   It tells me the distro and version. I need this to know package tooling, compatibility, and support/lifecycle context.

2. **What information do I get from `uname -r` / `cat /proc/version`, and why does semantic versioning matter?**
   `uname -r` gives a quick kernel release, while `cat /proc/version` gives fuller kernel build/version context. Semantic versioning (`X.Y.Z` style parts) helps separate major/minor/patch-level changes when troubleshooting or checking compatibility.

3. **Why do I care about uptime, and what are the last three numbers?**
   Uptime helps me understand stability and restart history. The last three numbers are 1-, 5-, and 15-minute load averages, which tell me if the system is cooling down or getting busier.

4. **What does `cat /proc/cmdline` show, and why does it matter for fleets?**
   It shows boot-time kernel parameters. For fleets, this matters because I can compare hosts and catch boot/config drift quickly.

5. **Is the system under high load? Why or why not?**
   The load averages from `uptime` (`0.78, 0.71, 0.65`) are well under capacity on an 8-core machine. `vmstat` and `mpstat` (covered in Digging Deeper) give sharper confirmation — breaking down memory pressure and per-interval CPU utilization respectively.

   > *Initial brainstorm:* I can do a quick glance with `htop` or `ps -elf | less` (or `ps -elf | vim -`). Then I compare what I see with uptime/load averages and CPU capacity.

6. **What user is running most processes, and why might that matter?**
   `ps -ef | awk '{print $1}' | sort | uniq -c` gives a clean per-user process count. On this system `root` dominated, which is expected — most background services run as root or a dedicated service account. I can also use `systemd-cgtop` and `systemd-cgls` to inspect ownership from the cgroup side, which helps distinguish expected service accounts from suspicious activity.

   > *Initial brainstorm:* My quick method from notes: `ps -elf | vim -` then `:g/[whatever your eyes see the most]/d` and use `:v` for the inverse to isolate likely majority owners.

7. **Why are networking errors useful during troubleshooting?**
   They help narrow down where the failure is happening (auth, route, DNS, service-side resets, etc.) instead of guessing.

## Digging Deeper

Now let's look at the system more closely using dedicated monitoring tools. Run each command and think about what the output is telling you. You may run them multiple times to compare snapshots.

1. Read `man ps` and identify additional useful options beyond what we already used.
2. Research a common Linux OS release and determine whether it has an LTS version — document what LTS means.

In [ ]:
# Virtual memory stats — args: interval (seconds), count
vmstat 1 5

# CPU utilization — same argument pattern
mpstat 1 5

# Process count per user
ps -ef | awk '{print $1}' | sort | uniq -c

# Per-process CPU usage over time — same argument pattern
pidstat 1 5

# CPU and disk I/O stats (-x extended, -z suppress idle rows) — same argument pattern
iostat -xz 1 5

### Output Snapshot (Digging Deeper)

```bash
$ vmstat 1 5
procs -----------memory---------- ---swap-- -----io---- -system-- -------cpu-------
 r  b   swpd   free   buff  cache   si   so    bi    bo   in   cs us sy id wa st gu
 3  0    100 5071688  2396 5634472    0    0   450   753 5791   15 11  2 86  1  0  0
 0  0    100 5074780  2396 5634848    0    0     0  2744 5177 15817  3  1 94  2  0  0
 1  0    100 5073136  2396 5634756    0    0     0     8 4113 11068  2  1 96  1  0  0
 4  0    100 5044176  2396 5665768    0    0     0    12 5402 13813  5  1 92  2  0  0
 3  0    100 5045472  2396 5655956    0    0     0    84 5153 13769  4  1 94  0  0  0
```

```bash
$ mpstat 1 5
Linux 6.18.9-2-cachyos (cachyos-x8664)  02/19/2026  _x86_64_  (8 CPU)

03:46:55 AM  CPU    %usr   %nice    %sys %iowait    %irq   %soft  %steal  %guest  %gnice   %idle
03:46:56 AM  all    4.77    0.00    1.00    1.25    0.25    0.13    0.00    0.00    0.00   92.60
03:46:57 AM  all    2.76    0.00    1.00    2.26    0.25    0.00    0.00    0.00    0.00   93.73
03:46:58 AM  all    3.14    0.00    0.88    0.88    0.38    0.00    0.00    0.00    0.00   94.73
03:46:59 AM  all   11.59    0.00    2.27    1.89    0.38    0.25    0.00    0.00    0.00   83.63
03:47:00 AM  all    6.27    0.00    1.25    0.75    0.50    0.00    0.00    0.00    0.00   91.22
Average:     all    5.70    0.00    1.28    1.41    0.35    0.08    0.00    0.00    0.00   91.19
```

```bash
$ ps -ef | awk '{print $1}' | sort | uniq -c
      2 avahi
      2 dbus
      1 polkitd
    207 root
      1 rtkit
     76 seydema+
      2 systemd+
      1 UID
```

```bash
$ pidstat 1 5
Linux 6.18.9-2-cachyos (cachyos-x8664)  02/19/2026  _x86_64_  (8 CPU)

03:47:00 AM   UID       PID    %usr %system  %guest   %wait    %CPU   CPU  Command
03:47:01 AM  1000      1072    2.97    1.98    0.00    0.00    4.95     7  Hyprland
03:47:01 AM  1000      1543   70.30    7.92    0.00    4.95   78.22     5  zen-bin
03:47:01 AM  1000      1671    0.99    1.98    0.00    0.00    2.97     1  RDD Process
03:47:01 AM  1000      1985    0.99    0.00    0.00    0.99    0.99     7  zellij
03:47:01 AM  1000      9647   38.61    4.95    0.00    3.96   43.56     6  Isolated Web Co
03:47:01 AM  1000     10638   24.75    2.97    0.00    0.99   27.72     2  claude
...
03:47:02 AM   UID       PID    %usr %system  %guest   %wait    %CPU   CPU  Command
03:47:02 AM  1000      1543   41.00    5.00    0.00    0.00   46.00     2  zen-bin
03:47:02 AM  1000      1985    1.00    1.00    0.00    0.00    2.00     1  zellij
03:47:02 AM  1000     10317    6.00    2.00    0.00    0.00    8.00     1  Isolated Web Co
03:47:02 AM  1000     10638   21.00    4.00    0.00    1.00   25.00     4  claude
...
Average:      UID       PID    %usr %system  %guest   %wait    %CPU   CPU  Command
Average:     1000      1072    1.80    2.20    0.00    0.00    3.99     -  Hyprland
Average:     1000      1543   34.33    4.59    0.00    1.20   38.92     -  zen-bin
Average:     1000      1985    1.00    0.40    0.00    0.20    1.40     -  zellij
Average:     1000      9647    7.98    1.00    0.00    0.80    8.98     -  Isolated Web Co
Average:     1000     10317   19.36    1.80    0.00    0.80   21.16     -  Isolated Web Co
Average:     1000     10638   21.36    2.79    0.00    0.40   24.15     -  claude
```

```bash
$ iostat -xz 1 5
Linux 6.18.9-2-cachyos (cachyos-x8664)  02/19/2026  _x86_64_  (8 CPU)

avg-cpu:  %user   %nice %system %iowait  %steal   %idle
          10.98    0.01    1.65    0.81    0.00   86.55

Device    r/s    rkB/s  rrqm/s  %rrqm r_await rareq-sz    w/s    wkB/s  wrqm/s  %wrqm w_await wareq-sz    d/s    dkB/s  %util
sda      0.05     0.82    0.00   0.00    9.88    17.85   0.00     0.00    0.00   0.00    0.00     0.00   0.00     0.00   0.05
sdb      9.31   447.86    1.46  13.54    1.38    48.13  18.78   759.45   14.02  42.74    3.86    40.43   5.34  1712.05   1.22
zram0    0.08     0.52    0.00   0.00    0.02     6.57   0.07     0.28    0.00   0.00    0.03     4.00   0.00     0.00   0.00

avg-cpu:  %user   %nice %system %iowait  %steal   %idle
           5.38    0.00    2.12    0.62    0.00   91.88

Device    r/s    rkB/s  rrqm/s  %rrqm r_await rareq-sz    w/s    wkB/s  wrqm/s  %wrqm w_await wareq-sz    d/s    dkB/s  %util
sdb      0.00     0.00    0.00   0.00    0.00     0.00   2.00     8.00    0.00   0.00    0.50     4.00   0.00     0.00   0.00
...
```

### Reflection: Digging Deeper

1. **`vmstat`: Is this system under high memory pressure?**
   `swpd` is 100 — essentially nothing, a negligible amount of swap in use. `free` sits around 5 GB, and `si`/`so` are both 0 (no pages swapping in or out). The system is not under memory pressure.

2. **`mpstat`: Is this system under high CPU load?**
   `%idle` ranges from 83–96% across the five samples, averaging ~91%. There's a noticeable spike at 03:46:59 (idle dropped to 83%) but it quickly recovered. The system is being actively used but is not under high CPU load.

3. **`ps -ef | awk ... | uniq -c`: Which user owns the most processes? Is the system doing real work?**
   `root` owns 207 processes — still the majority, expected for a running OS. But `seydema+` has 76, which is notably high and reflects an active user session (browser, compositor, terminal multiplexer, etc.). This system is clearly doing real work, not just sitting idle.

4. **`pidstat`: Why does output length vary between intervals? Which processes showed up most?**
   `pidstat` only reports processes that consumed measurable CPU in that interval — silent intervals produce fewer rows. `zen-bin` (Firefox) dominated with up to 78% CPU in the first interval, then settled lower. The `claude` process also appeared consistently at 21–27%, and various `Isolated Web Co` browser tab processes came and went throughout.

5. **`iostat`: How would you modify this to run for 30 seconds instead of 5?**
   Change the count argument: `iostat -xz 1 30`. The first block is a cumulative average since boot — every block after that reflects just that interval. `sdb` is the active drive here; `sda` is barely touched, and `zram0` shows minimal compressed memory swap activity.